# England Cities — Prescription Map
Step-by-step map visualisation using Plotly and Pandas.

## Step 1 — Imports

In [58]:
import pandas as pd
import requests
import plotly.graph_objects as go

## Step 2 — City coordinates

In [59]:
coords = {
    "Birmingham":          (52.4862, -1.8904),
    "Brighton":            (50.8225, -0.1372),
    "Bristol":             (51.4545, -2.5879),
    "Canterbury":          (51.2802,  1.0789),
    "Exeter":              (50.7184, -3.5339),
    "Leeds":               (53.8008, -1.5491),
    "London":              (51.5074, -0.1278),
    "Manchester":          (53.4808, -2.2426),
    "Middlesbrough":       (54.5742, -1.2350),
    "Newcastle upon Tyne": (54.9783, -1.6178),
    "Nottingham":          (52.9548, -1.1581),
    "Peterborough":        (52.5695, -0.2405),
    "Plymouth":            (50.3755, -4.1427),
    "Sunderland":          (54.9061, -1.3838),
    "York":                (53.9590, -1.0815),
}

df_coords = pd.DataFrame(
    [(city, lat, lon) for city, (lat, lon) in coords.items()],
    columns=["city", "lat", "lon"]
)
df_coords

,city,lat,lon
0,Birmingham,52.4862,-1.8904
1,Brighton,50.8225,-0.1372
2,Bristol,51.4545,-2.5879
3,Canterbury,51.2802,1.0789
4,Exeter,50.7184,-3.5339
5,Leeds,53.8008,-1.5491
6,London,51.5074,-0.1278
7,Manchester,53.4808,-2.2426
8,Middlesbrough,54.5742,-1.2350
9,Newcastle upon Tyne,54.9783,-1.6178


## Step 3 — Load prescription data

In [60]:
df_city = pd.read_csv("../EDA/data/England_all_per_city.csv")
df_city.columns = df_city.columns.str.strip()
df_city["date"] = pd.to_datetime(df_city["date"])
df_city["year"] = df_city["date"].dt.year

df_rx = (
    df_city[df_city["group"] == "antidepressant"]
    .groupby(["year", "city"])["items_per_1k_per_day"]
    .sum()
    .reset_index()
    .rename(columns={"items_per_1k_per_day": "rx"})
)

df_rx[df_rx["year"] == 2023].sort_values("rx", ascending=False)

,year,city,rx
40,2023,Nottingham,115.77
37,2023,Manchester,113.62
38,2023,Middlesbrough,108.29
39,2023,Newcastle upon Tyne,103.33
32,2023,Bristol,78.58
41,2023,Peterborough,68.82
43,2023,Sunderland,67.49
34,2023,Exeter,65.69
44,2023,York,62.01
42,2023,Plymouth,60.38


## Step 4 — Load city boundaries + England border

14 cities from GitHub LAD GeoJSON. London from ONS. England outline from ONS Countries layer.

In [61]:
NAME_MAP = {
    "Birmingham":          "Birmingham",
    "Brighton":            "Brighton and Hove",
    "Bristol":             "Bristol, City of",
    "Canterbury":          "Canterbury",
    "Exeter":              "Exeter",
    "Leeds":               "Leeds",
    "Manchester":          "Manchester",
    "Middlesbrough":       "Middlesbrough",
    "Newcastle upon Tyne": "Newcastle upon Tyne",
    "Nottingham":          "Nottingham",
    "Peterborough":        "Peterborough",
    "Plymouth":            "Plymouth",
    "Sunderland":          "Sunderland",
    "York":                "York",
}

# ── City LAD boundaries ───────────────────────────────────────────────────────
print("Downloading LAD boundaries...")
lad_url = "https://raw.githubusercontent.com/martinjc/UK-GeoJSON/master/json/administrative/eng/lad.json"
r = requests.get(lad_url, timeout=30)
print(f"  Status: {r.status_code}   Size: {len(r.content)/1024:.0f} KB")
all_lad = r.json()["features"]

props = all_lad[0]["properties"]
name_field = next((k for k in props if k.upper().endswith("NM") or k.lower() == "name"), None)
print(f"  Name field: '{name_field}'")

reverse = {v: k for k, v in NAME_MAP.items()}
features = []
for feat in all_lad:
    city = reverse.get(feat["properties"].get(name_field, ""))
    if city:
        feat["id"] = city
        feat["properties"]["city"] = city
        features.append(feat)
        print(f"  OK  {city}")

# ── London from ONS Regions ───────────────────────────────────────────────────
print("Fetching London boundary...")
rgn_url = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Regions_December_2022_EN_BUC/FeatureServer/0/query"
)
r_lon = requests.get(rgn_url, params={"where": "RGN22NM='London'", "outFields": "RGN22NM", "f": "geojson", "outSR": "4326"})
for feat in r_lon.json().get("features", []):
    feat["id"] = "London"
    feat["properties"]["city"] = "London"
    features.append(feat)
    print("  OK  London")

geojson = {"type": "FeatureCollection", "features": features}
print(f"  Total: {len(features)}/15 city boundaries loaded")

# ── England outline from ONS Countries ───────────────────────────────────────
print("Fetching England border...")
ctr_url = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Countries_December_2022_UK_BUC/FeatureServer/0/query"
)
r_eng = requests.get(ctr_url, params={"where": "CTRY22NM='England'", "outFields": "CTRY22NM", "f": "geojson", "outSR": "4326"})
print(f"  Status: {r_eng.status_code}   Size: {len(r_eng.content)/1024:.0f} KB")
england_geo = r_eng.json()

def extract_rings(geometry):
    """Return list of (lats, lons) for each ring in a Polygon or MultiPolygon."""
    rings = []
    if geometry["type"] == "Polygon":
        polygons = [geometry["coordinates"]]
    else:
        polygons = geometry["coordinates"]
    for poly in polygons:
        for ring in poly:
            lons = [c[0] for c in ring]
            lats = [c[1] for c in ring]
            rings.append((lats, lons))
    return rings

england_rings = []
for feat in england_geo.get("features", []):
    england_rings += extract_rings(feat["geometry"])
print(f"  England outline: {len(england_rings)} ring(s)")

# ── All 9 England regions ─────────────────────────────────────────────────────
print("Fetching all England regions...")
r_rgn = requests.get(rgn_url, params={"where": "1=1", "outFields": "RGN22NM", "f": "geojson", "outSR": "4326"})
print(f"  Status: {r_rgn.status_code}   Size: {len(r_rgn.content)/1024:.0f} KB")
region_rings = []
for feat in r_rgn.json().get("features", []):
    region_rings += extract_rings(feat["geometry"])
print(f"  Regions: {len(region_rings)} ring(s)")


  Status: 200   Size: 9411 KB
  Name field: 'LAD13NM'
  OK  Middlesbrough
  OK  York
  OK  Nottingham
  OK  Bristol
  OK  Plymouth
  OK  Peterborough
  OK  Brighton
  OK  Exeter
  OK  Canterbury
  OK  Manchester
  OK  Newcastle upon Tyne
  OK  Sunderland
  OK  Birmingham
  OK  Leeds
Fetching London boundary...
  OK  London
  Total: 15/15 city boundaries loaded
Fetching England border...
  Status: 200   Size: 118 KB
  England outline: 36 polygon ring(s)


## Step 5 — Choropleth map: prescription rate by city

Gill Sans labels above each city, England border outline, pink→red gradient. Change `YEAR` to switch years.

In [66]:
YEAR = 2023   # ← change to 2021–2025

rx_year = df_rx[df_rx["year"] == YEAR].set_index("city")["rx"]
cities  = [f["id"] for f in geojson["features"]]
z_vals  = [round(rx_year.get(c, 0), 1) for c in cities]

label_df = df_coords[df_coords["city"].isin(cities)].copy()
label_df = label_df.merge(pd.DataFrame({"city": cities, "rx": z_vals}), on="city")

fig3 = go.Figure()

# ── Choropleth ────────────────────────────────────────────────────────────────
fig3.add_trace(go.Choropleth(
    geojson=geojson,
    locations=cities,
    z=z_vals,
    colorscale=[[0, "#FFBBBB"], [0.5, "#E83A3A"], [1, "#6C0000"]],
    zmin=min(z_vals),
    zmax=max(z_vals),
    colorbar=dict(
        title=dict(text="Rx / 1k / day", font=dict(size=12, color="#333")),
        tickfont=dict(color="#333"),
        thickness=14,
        len=0.6,
    ),
    marker=dict(line=dict(color="white", width=1.2)),
    hovertemplate="<b>%{location}</b><br>%{z} Rx / 1k / day<extra></extra>",
))

# ── England border outline ────────────────────────────────────────────────────
for lats, lons in england_rings:
    fig3.add_trace(go.Scattergeo(
        lat=lats,
        lon=lons,
        mode="lines",
        line=dict(color="#AFAFAF", width=1.5),
        hoverinfo="skip",
        showlegend=False,
    ))


# ── Region outlines (thin, light grey) ──────────────────────────────────────
for lats, lons in region_rings:
    fig3.add_trace(go.Scattergeo(
        lat=lats,
        lon=lons,
        mode="lines",
        line=dict(color="#AAAAAA", width=0.6),
        hoverinfo="skip",
        showlegend=False,
    ))
# ── City labels above polygon ─────────────────────────────────────────────────
fig3.add_trace(go.Scattergeo(
    lat=label_df["lat"],
    lon=label_df["lon"],
    text=label_df["city"],
    mode="markers+text",
    textposition="top center",
    marker=dict(size=4, color="#333333", symbol="circle"),
    textfont=dict(size=13, color="#333333", family="Trebuchet MS, Impact, Arial Narrow, sans-serif"),
    hoverinfo="skip",
    showlegend=False,
))

fig3.update_layout(
    title=dict(
        text=f"Antidepressant Prescriptions per 1,000 Population — England {YEAR}",
        x=0.5,
        font=dict(size=15, color="#222", family="Gill Sans, Helvetica Neue, Arial, sans-serif"),
    ),
    geo=dict(
        scope="europe",
        resolution=50,
        lonaxis=dict(range=[-6.5, 2.5]),
        lataxis=dict(range=[49.5, 56.5]),
        showland=True,
        landcolor="#F2EFE9",
        showocean=True,
        oceancolor="#A8D8EA",
        showcountries=False,
        showcoastlines=False,
        showlakes=True,
        lakecolor="#A8D8EA",
        bgcolor="white",
    ),
    paper_bgcolor="white",
    font=dict(color="#333", family="Gill Sans, Helvetica Neue, Arial, sans-serif"),
    margin=dict(t=60, b=10, l=10, r=10),
    height=680,
)

fig3.show()
